# Modèle ZÉRO : français + kirundi, entraîné depuis le début

Ce modèle part de **rien** (pas de Pythia, poids au hasard, nouveau tokenizer).

| Étape | Ce qui se passe | Durée (GPU T4) |
|---|---|---|
| 0 | Préparer Colab : GPU, Google Drive, code | 2 min |
| 1 | Télécharger le français (Wikipédia FR + conversations OpenAssistant) | 30 à 60 min |
| 2 | Créer le tokenizer (32 000 morceaux, moitié kirundi, moitié français) | 15 min |
| 3 | **Lecture** : le modèle lit tous les textes | 12 à 15 h (plusieurs sessions) |
| 4 | **Discussion** : il apprend à répondre (salutations, questions, traductions) | 1 à 2 h |
| 5 | Tests | 10 min |
| 6 | Essayer et télécharger le `.zip` | |

**Avant de commencer** : sur ton ordinateur, glisse le dossier `A_METTRE_DANS_DRIVE/donnees_zero` dans **Mon Drive** (à la racine).

**Si Colab coupe** : relance l'étape 0, puis la cellule de l'étape en cours. Tout reprend au dernier point sauvegardé.

## Étape 0 : préparer Colab

In [ ]:
# 0.1  Vérifier le GPU (menu : Exécution > Modifier le type d'exécution > GPU T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 0.2  Brancher Google Drive : TOUT est sauvegardé là (données, modèles)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0.3  Récupérer le code depuis GitHub
#      Ton dépôt : NiBienvenu/modele_train.
#      Dépôt privé : mets un token dans les Secrets de Colab (icône clé) sous le nom GH_TOKEN.
DEPOT = "https://github.com/NiBienvenu/modele_train.git"

import os
if not os.path.exists("/content/modele_train"):
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        DEPOT = DEPOT.replace("https://", f"https://{token}@")
    except Exception:
        pass  # dépôt public
    !git clone -q $DEPOT /content/modele_train
else:
    !cd /content/modele_train && git pull -q
%cd /content/modele_train
!pip install -q -r requirements.txt

In [ ]:
# 0.4  Les dossiers utilisés partout (sur Google Drive)
import os
CODE = "/content/modele_train"
def aller_au_code():
    if not os.path.exists(f"{CODE}/train.py"):
        raise SystemExit("Le code n'est pas dans " + CODE + " : relance la cellule 0.3.")
    os.chdir(CODE)
aller_au_code()

RACINE = "/content/drive/MyDrive"
Z  = f"{RACINE}/donnees_zero"     # données (venues de ton ordinateur + Wikipédia FR)
MZ = f"{RACINE}/modele_zero"      # modèles
os.makedirs(f"{MZ}/journaux", exist_ok=True)
for chemin in (f"{Z}/lecture_rn", f"{Z}/lecture_bilingue", f"{Z}/grammaire", f"{Z}/dialogues/salutations_rn.jsonl"):
    print(("OK      " if os.path.exists(chemin) else "MANQUANT"), chemin)

def fait(chemin):
    return os.path.exists(chemin)

def verifier(chemin, etape):
    if os.path.exists(chemin):
        print(f"OK : {etape} terminée -> {chemin}")
        return
    raise SystemExit(f"ÉCHEC : {etape} n'a pas produit {chemin}. Regarde {MZ}/journaux/{etape}.txt. "
                     "Ne passe PAS à l'étape suivante.")

def meilleur_modele():
    for ph in ("chat", "lecture"):
        for sous in ("finale", "meilleure", "derniere"):
            if os.path.exists(f"{MZ}/{ph}/{sous}/config.json"):
                return f"{MZ}/{ph}/{sous}"
    raise SystemExit("Aucun modèle trouvé : lance d'abord l'étape 3.")

In [ ]:
# 0.5  OÙ EN SUIS-JE ?
for nom, chemin in [("Français téléchargé", f"{Z}/lecture_fr/wikipedia_fr_001.txt"),
                    ("Tokenizer", f"{MZ}/tokenizer/tokenizer.json"),
                    ("Lecture", f"{MZ}/lecture/finale"), ("Discussion", f"{MZ}/chat/finale")]:
    reprise = os.path.exists(chemin.replace("finale", "derniere"))
    print(f"{nom:22s}: " + ("FINI" if fait(chemin) else "en cours (reprise possible)" if reprise else "pas commencé"))

## Étape 1 : télécharger le français

In [ ]:
aller_au_code()
if not fait(f"{Z}/dialogues/oasst_fr.jsonl") or not fait(f"{Z}/lecture_fr/wikipedia_fr_001.txt"):
    !python -u collecte/telecharger_fr.py --sortie $Z --mots 150000000 2>&1 | tee $MZ/journaux/francais.txt
!ls -la $Z/lecture_fr | head

## Étape 2 : le tokenizer

Le kirundi est lu 10 fois et un seul fichier Wikipédia (20 millions de mots) une fois : les deux langues ont à peu près **la même place** dans le vocabulaire.
Regarde la vérification à la fin : les phrases kirundi doivent être coupées en morceaux qui ont du sens (préfixes, racines).

In [ ]:
aller_au_code()
if not fait(f"{MZ}/tokenizer/tokenizer.json"):
    !python train_tokenizer.py --taille_vocab 32000 \
        --source "$Z/lecture_rn/*.txt=10" --source "$Z/lecture_bilingue/*.txt=3" \
        --source "$Z/lecture_fr/wikipedia_fr_001.txt=1" --source "$Z/lecture_fr/bible_fr.txt=1" \
        --sortie $MZ/tokenizer 2>&1 | tee $MZ/journaux/tokenizer.txt
verifier(f"{MZ}/tokenizer/tokenizer.json", "tokenizer")

## Étape 3 : lecture (la plus longue)

**Si Colab coupe, relance l'étape 0 puis cette cellule** : ça reprend tout seul.
Si *CUDA out of memory* : dans `configs/zero_pretrain.yaml`, mets `taille_lot: 4` et `accumulation: 32`.

In [ ]:
aller_au_code()
if not fait(f"{MZ}/lecture/finale"):
    !python -u train.py --config configs/zero_pretrain.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $MZ/journaux/lecture.txt
verifier(f"{MZ}/lecture/finale", "lecture")

## Étape 4 : discussion (apprendre à répondre)

In [ ]:
aller_au_code()
if not fait(f"{MZ}/chat/finale"):
    !python -u train.py --config configs/zero_chat.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a $MZ/journaux/chat.txt
verifier(f"{MZ}/chat/finale", "chat")

## Étape 5 : tests (jamais vus à l'entraînement)

In [ ]:
aller_au_code()
!python -u evaluer.py --modele {meilleur_modele()} --tests tests/dialogues_test.jsonl --montrer 3 \
    --sortie $MZ/journaux/resultats_tests.jsonl 2>&1 | tee $MZ/journaux/notes_tests.txt

## Étape 6 : essayer le modèle et le télécharger

In [ ]:
aller_au_code()
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
appareil = "cuda" if torch.cuda.is_available() else "cpu"
dossier = meilleur_modele()
tok = AutoTokenizer.from_pretrained(dossier)
model = AutoModelForCausalLM.from_pretrained(dossier).to(appareil).eval()
historique = ""

def demander(question, garder=True):
    """Pose une question. garder=True : le modèle se souvient des tours d'avant."""
    global historique
    prompt = f"{historique}<|user|> {question} <|assistant|>"
    ids = tok(prompt, return_tensors="pt", return_token_type_ids=False).to(appareil)
    out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.3, top_p=0.9,
                         eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    reponse = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if garder:
        historique = f"{prompt} {reponse}<|endoftext|>"
    return reponse

print(dossier)
for q in ["Bonjour", "Bwakeye.", "N'amaki?", "Comment dit-on merci en kirundi ?", "Qui es-tu ?"]:
    print(q, "->", demander(q, garder=False))

In [ ]:
# Une vraie conversation (historique = "" pour recommencer)
historique = ""
print(demander("Bonjour"))
print(demander("Comment dit-on au revoir en kirundi ?"))

In [ ]:
# Le .zip pour le backend (le mettre dans litgpt/models/local/ puis scripts/convert_local_model.py)
dossier_modele = meilleur_modele()
!cd "$dossier_modele" && zip -qr /content/modele_zero_fr_rn.zip . -x "etat_entrainement.pt" && ls -lh /content/modele_zero_fr_rn.zip
!cp /content/modele_zero_fr_rn.zip $MZ/
from google.colab import files
files.download("/content/modele_zero_fr_rn.zip")